In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

inp = Path("/kaggle/input")

print("=== DATASET-ET ===")
for p in inp.iterdir():
    print(p)

print("\n=== FAJLLAT E PROJEKTIT ===")
for name in [
    "strong_cli.py",
    "submission_matching-i.json",
    "submission_matching-ii.json"
]:
    found = list(inp.rglob(name))
    print(f"\n{name}: {len(found)}")
    for p in found[:3]:
        print(p)

print("\n=== ZIP FAJLLAT ===")
for p in inp.rglob("*.zip"):
    print(p)

In [ ]:
from pathlib import Path
import shutil
import os
import sys

# Backup-i i gjetur ne Kaggle
source = Path(
    "/kaggle/input/datasets/oltamorina/esa-spoc4-full-backup"
)

# Folderi ku do te punojme
root = Path("/kaggle/working/Luna-Tomato-Logistics")

# Kontrollo backup-in
assert (source / "src/strong_cli.py").exists(), "Mungon kodi!"
assert (source / "results/submission_matching-i.json").exists()
assert (source / "results/submission_matching-ii.json").exists()

# Kopjo projektin ne working
shutil.copytree(
    source,
    root,
    dirs_exist_ok=True
)

os.chdir(root)
sys.path.insert(0, str(root))

print("PROJEKTI U RIKTHYE ME SUKSES!")
print("STRONG OPTIMIZER: OK")
print("MATCHING I: OK")
print("MATCHING II: OK")

In [ ]:
from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

for name in ["matching-i", "matching-ii"]:

    rows = load_transfers(f"data/{name}.txt")

    chosen = read_incumbent(
        f"results/submission_{name}.json",
        len(rows),
        name
    )

    assert chosen is not None, f"Zgjidhja {name} nuk u gjet!"

    score = score_solution(rows, sorted(chosen))

    print(f"{name}: {score:.3f} | VALID LOKAL")

print("GATI PER OPTIMIZIM!")

In [ ]:
from pathlib import Path
from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

for name in ["matching-i", "matching-ii"]:

    rows = load_transfers(f"data/{name}.txt")

    chosen = read_incumbent(
        Path(f"results/submission_{name}.json"),
        len(rows),
        name
    )

    assert chosen is not None, f"Zgjidhja {name} nuk u gjet!"

    score = score_solution(rows, sorted(chosen))

    print(f"{name}: {score:.3f} | VALID LOKAL")

print("GATI PER OPTIMIZIM!")

In [ ]:
from pathlib import Path
import os, sys, shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

for name in ["matching-i", "matching-ii"]:
    rows = load_transfers(root / "data" / f"{name}.txt")
    file = root / "results" / f"submission_{name}.json"

    chosen = read_incumbent(file, len(rows), name)
    assert chosen is not None, f"Gabim te {name}"

    score = score_solution(rows, sorted(chosen))
    print(f"{name}: {score:.3f} | VALID LOKAL")

# Ruajme Matching II para optimizimit
shutil.copy2(
    root / "results/submission_matching-ii.json",
    root / "results/matching_ii_before_cpsat.json"
)

print("PROJEKTI ESHTE GATI!")

In [ ]:
try:
    from ortools.sat.python import cp_model
    print("CP-SAT ESHTE GATI!")
except ImportError:
    print("DUHET INSTALUAR OR-TOOLS")

In [ ]:
%pip install -q ortools

In [ ]:
try:
    import ortools
    from ortools.sat.python import cp_model

    print("OR-TOOLS VERSION:", ortools.__version__)
    print("CP-SAT ESHTE GATI!")
except Exception as e:
    print("GABIM:", type(e).__name__, str(e))

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.strong_cli \
  --input data/matching-ii.txt \
  --problem matching-ii \
  --seconds 2 \
  --rounds 1 \
  --seed 2050 \
  --neighborhoods 95 \
  --candidate-caps 1000 \
  --per-round 1.0 \
  --cpsat-seconds 120 \
  --workers 2

In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")
results = root / "results"

shutil.copy2(
    results / "submission_matching-ii.json",
    results / "BEST_MATCHING_II_70678.json"
)

print("REZULTATI I RI U RUAJT!")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.strong_cli \
  --input data/matching-i.txt \
  --problem matching-i \
  --seconds 300 \
  --rounds 6000 \
  --seed 2053 \
  --neighborhoods 85,160,350,550 \
  --candidate-caps 1200,2600,5000,8000 \
  --per-round 1.5

In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")
source = root / "results/submission_matching-i.json"
backup = root / "results/BEST_MATCHING_I_32934.json"

assert source.exists(), "Nuk u gjet rezultati!"
shutil.copy2(source, backup)

print("REKORDI 32934.585 U RUAJT!")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.strong_cli \
  --input data/matching-i.txt \
  --problem matching-i \
  --seconds 360 \
  --rounds 4500 \
  --seed 2079 \
  --neighborhoods 350,550,750,950 \
  --candidate-caps 5000,8000,11000,14000 \
  --per-round 2.0

In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")

shutil.copy2(
    root / "results/submission_matching-i.json",
    root / "results/BEST_MATCHING_I_33055.json"
)

print("REKORDI 33055.107 U RUAJT!")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.strong_cli \
  --input data/matching-i.txt \
  --problem matching-i \
  --seconds 360 \
  --rounds 3000 \
  --seed 2087 \
  --neighborhoods 550,950,1300,1600 \
  --candidate-caps 8000,14000,17000,20000 \
  --per-round 2.0

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil, os

root = Path("/kaggle/working/Luna-Tomato-Logistics")
inp = Path("/kaggle/input")

assert (root / "src/strong_opt.py").exists(), "Rikthe projektin!"

names = [
    "src/focused_opt.py",
    "src/focused_cli.py",
    "tests/test_focused.py"
]

sources = [
    p.parent.parent
    for p in inp.rglob("focused_opt.py")
    if p.parent.name == "src"
]

if sources:
    for name in names:
        dst = root / name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(sources[0] / name, dst)
else:
    archives = [
        p for p in inp.rglob("*.zip")
        if "FOCUSED" in p.name.upper()
    ]
    assert archives, "Mungon Dataset-i Focused!"

    with ZipFile(archives[0]) as z:
        for name in names:
            dst = root / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(z.read(name))

os.chdir(root)
print("FOCUSED OPTIMIZER U INSTALUA!")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m unittest discover -s tests -p 'test_focused.py' -v

In [ ]:
!python -m src.focused_cli \
  --input data/matching-i.txt \
  --problem matching-i \
  --method cpsat \
  --seconds 180 \
  --rounds 75 \
  --sizes 120,260,450 \
  --caps 1000,2000,3500 \
  --per-round 2.5 \
  --workers 2 \
  --seed 2099

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics
!python -m unittest discover -s tests -p 'test_focused.py' -v

In [ ]:
!python -m src.focused_cli \
  --input data/matching-i.txt \
  --problem matching-i \
  --method cpsat \
  --seconds 240 \
  --rounds 40 \
  --sizes 550,900,1400 \
  --caps 4000,6500,9000 \
  --per-round 8 \
  --workers 2 \
  --seed 2177

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil, os

root = Path("/kaggle/working/Luna-Tomato-Logistics")
inp = Path("/kaggle/input")

names = [
    "src/focused_opt.py",
    "src/focused_cli.py",
    "tests/test_focused_v3.py"
]

assert (root / "src/strong_opt.py").exists(), "Projekti mungon!"

backup = root / "backup_focused_v2"
backup.mkdir(exist_ok=True)

for name in names[:2]:
    old = root / name
    if old.exists():
        shutil.copy2(old, backup / old.name)

found = list(inp.rglob("test_focused_v3.py"))

if found:
    source = found[0].parent.parent
    for name in names:
        dest = root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / name, dest)
else:
    zips = list(inp.rglob("ESA_SpOC4_FOCUSED_V3.zip"))
    assert zips, "Shto Dataset-in V3 te Add Input!"

    with ZipFile(zips[0]) as z:
        for name in names:
            dest = root / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(name))

os.chdir(root)
print("FOCUSED V3 U INSTALUA!")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m unittest discover -s tests -p 'test_focused*.py' -v

In [ ]:
!python -m src.focused_cli \
  --input data/matching-i.txt \
  --problem matching-i \
  --method hybrid \
  --seconds 120 \
  --rounds 35 \
  --sizes 260,450,650 \
  --caps 1600,3000,4500 \
  --per-round 3 \
  --workers 2 \
  --seed 2217

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, adaptive_lns
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-i"
target = root / "results" / f"submission_{name}.json"

rows = load_transfers(root / "data" / f"{name}.txt")
best = read_incumbent(target, len(rows), name)

assert best is not None, "Mungon zgjidhja!"
best_score = score_solution(rows, sorted(best))
p = pack(rows)

# Ruajme rezultatin aktual
backup = target.parent / f"before_restarts_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}")

rng = random.Random(2307)

def shake(selected, n_drop):
    removed = set(
        rng.sample(sorted(selected), min(n_drop, len(selected)))
    )
    kept = set(selected) - removed
    oe, ol, od = owner_maps(p, kept)

    pool = set(removed)

    for j in removed:
        pool.update(p.e_rows[p.e[j]])
        pool.update(p.l_rows[p.l[j]])
        pool.update(p.d_rows[p.d[j]])

    alternatives = [
        int(j) for j in pool
        if j not in selected and p.w[j] > 0
    ]

    priorities = [
        (float(p.w[j]) * rng.uniform(0.65, 1.35), j)
        for j in alternatives
    ]
    priorities.sort(reverse=True)

    # Provojme transferime alternative fillimisht
    for _, j in priorities:
        if (oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0):

            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    # Rikthejme transferimet origjinale nese ka vend
    for j in sorted(removed, key=lambda k: -p.w[k]):
        if (oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0):

            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    score_solution(rows, sorted(kept))
    return kept

for trial, n_drop in enumerate([80, 160, 260], 1):

    print(f"\n===== RESTART {trial}/3 =====", flush=True)

    starting = shake(best, n_drop)
    starting_score = score_solution(rows, sorted(starting))

    print(f"Alternative start: {starting_score:.3f}", flush=True)

    candidate, info = adaptive_lns(
        rows,
        starting,
        seconds=75,
        rounds=2000,
        seed=2307 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    score = score_solution(rows, sorted(candidate))

    print(f"Rezultati i restartit: {score:.3f}", flush=True)
    print(f"Rekordi: {best_score:.3f}", flush=True)

    if score > best_score + 1e-7:

        disk = read_incumbent(target, len(rows), name)
        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:
            document = [{
                "decisionVector": make_vector(
                    len(rows), sorted(candidate)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = candidate
            best_score = score

            print("REKORD I RI U RUAJT!", flush=True)
    else:
        print("Pa rekord te ri.", flush=True)

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")

shutil.copy2(
    root / "results/submission_matching-i.json",
    root / "results/BEST_MATCHING_I_33155.json"
)

zip_path = shutil.make_archive(
    "/kaggle/working/ESA_BACKUP_AFTER_RESTARTS",
    "zip",
    root_dir="/kaggle/working",
    base_dir="Luna-Tomato-Logistics"
)

print("REKORDI 33155.153 U RUAJT!")
print("BACKUP:", zip_path)

In [ ]:
from pathlib import Path
import shutil
import zipfile

root = Path("/kaggle/working/Luna-Tomato-Logistics")
assert root.exists(), "Folderi i projektit nuk u gjet!"

backup = shutil.make_archive(
    "/kaggle/working/ESA_PROJEKTI_FINAL_BACKUP",
    "zip",
    root_dir="/kaggle/working",
    base_dir="Luna-Tomato-Logistics"
)

with zipfile.ZipFile(backup) as z:
    assert z.testzip() is None, "ZIP-i ka problem!"
    for name in ["matching-i", "matching-ii"]:
        path = f"Luna-Tomato-Logistics/results/submission_{name}.json"
        assert path in z.namelist(), f"Mungon {path}"

print("BACKUP U KRIJUA ME SUKSES!")
print("Emri:", Path(backup).name)
print("Madhesia:", round(Path(backup).stat().st_size / 1024 / 1024, 2), "MB")

In [ ]:
from pathlib import Path
import os
import sys

root = Path("/kaggle/working/Luna-Tomato-Logistics")

assert root.exists(), (
    "Projekti nuk eshte ne working! "
    "Duhet ta rikthejme nga backup-i."
)

os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

scores = {}

for name in ["matching-i", "matching-ii"]:

    rows = load_transfers(root / "data" / f"{name}.txt")

    chosen = read_incumbent(
        root / "results" / f"submission_{name}.json",
        len(rows),
        name
    )

    assert chosen is not None, f"Mungon zgjidhja {name}"

    score = score_solution(rows, sorted(chosen))
    scores[name] = score

    print(f"{name}: {score:.3f} | VALID LOKAL")

if scores["matching-i"] >= 33155.15 and scores["matching-ii"] >= 70678.78:
    print("\nPROJEKTI ESHTE GATI PER VAZHDIM!")
else:
    print("\nKUJDES: Duhet kontrolluar backup-i i fundit!")

In [ ]:
from pathlib import Path
import os
import sys

root = Path("/kaggle/working/Luna-Tomato-Logistics")

assert root.exists(), (
    "Projekti nuk eshte ne working! "
    "Duhet ta rikthejme nga backup-i."
)

os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

scores = {}

for name in ["matching-i", "matching-ii"]:

    rows = load_transfers(root / "data" / f"{name}.txt")

    chosen = read_incumbent(
        root / "results" / f"submission_{name}.json",
        len(rows),
        name
    )

    assert chosen is not None, f"Mungon zgjidhja {name}"

    score = score_solution(rows, sorted(chosen))
    scores[name] = score

    print(f"{name}: {score:.3f} | VALID LOKAL")

if scores["matching-i"] >= 33155.15 and scores["matching-ii"] >= 70678.78:
    print("\nPROJEKTI ESHTE GATI PER VAZHDIM!")
else:
    print("\nKUJDES: Duhet kontrolluar backup-i i fundit!")

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import sys

inp = Path("/kaggle/input")
working = Path("/kaggle/working")

# Kerkojme backup-in e fundit
backups = [
    "ESA_PROJEKTI_FINAL_BACKUP.zip",
    "ESA_BACKUP_AFTER_RESTARTS.zip"
]

found = []

for name in backups:
    found.extend(inp.rglob(name))
    found.extend(working.glob(name))

if not found:
    print("BACKUP-I I FUNDIT NUK U GJET!")
    print("Duhet ta shtosh nga Add Input.")
else:
    backup = found[0]
    print("BACKUP U GJET:", backup)

    with ZipFile(backup) as z:
        assert z.testzip() is None, "ZIP-i eshte i demtuar!"
        z.extractall(working)

    root = working / "Luna-Tomato-Logistics"
    os.chdir(root)
    sys.path.insert(0, str(root))

    from src.matching import load_transfers, score_solution
    from src.cli import read_incumbent

    print("\nPROJEKTI U RIKTHYE!\n")

    for name in ["matching-i", "matching-ii"]:
        rows = load_transfers(root / "data" / f"{name}.txt")
        chosen = read_incumbent(
            root / "results" / f"submission_{name}.json",
            len(rows),
            name
        )
        assert chosen is not None

        score = score_solution(rows, sorted(chosen))
        print(f"{name}: {score:.3f} | VALID LOKAL")

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, adaptive_lns
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-ii"
target = root / "results" / f"submission_{name}.json"

rows = load_transfers(root / "data" / f"{name}.txt")
best = read_incumbent(target, len(rows), name)

assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))
p = pack(rows)

# Ruaje rekordin para eksperimentit
backup = target.parent / f"before_restarts_{name}_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

rng = random.Random(2407)

def shake(selected, n_drop):
    removed = set(
        rng.sample(sorted(selected), min(n_drop, len(selected)))
    )

    kept = set(selected) - removed
    oe, ol, od = owner_maps(p, kept)

    pool = set(removed)

    for j in removed:
        pool.update(p.e_rows[p.e[j]])
        pool.update(p.l_rows[p.l[j]])
        pool.update(p.d_rows[p.d[j]])

    alternatives = [
        int(j) for j in pool
        if j not in selected and p.w[j] > 0
    ]

    priorities = [
        (float(p.w[j]) * rng.uniform(0.65, 1.35), j)
        for j in alternatives
    ]
    priorities.sort(reverse=True)

    for _, j in priorities:
        if (oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0):

            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    for j in sorted(removed, key=lambda k: -p.w[k]):
        if (oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0):

            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    score_solution(rows, sorted(kept))
    return kept

# 3 eksperimente me nisje alternative
for trial, n_drop in enumerate([180, 350, 600], 1):

    print(f"\n===== RESTART {trial}/3 =====", flush=True)

    starting = shake(best, n_drop)
    starting_score = score_solution(rows, sorted(starting))

    print(f"Alternative start: {starting_score:.3f}", flush=True)

    candidate, info = adaptive_lns(
        rows,
        starting,
        seconds=100,
        rounds=2500,
        seed=2407 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    score = score_solution(rows, sorted(candidate))

    print(f"Rezultati i restartit: {score:.3f}", flush=True)
    print(f"Rekordi: {best_score:.3f}", flush=True)

    if score > best_score + 1e-7:

        disk = read_incumbent(target, len(rows), name)
        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:
            document = [{
                "decisionVector": make_vector(
                    len(rows), sorted(candidate)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = candidate
            best_score = score

            print("REKORD I RI U RUAJT!", flush=True)
    else:
        print("Pa rekord te ri.", flush=True)

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, adaptive_lns
from src.cli import read_incumbent, save_atomic, CHALLENGE

# ==========================================
# MATCHING II - DIVERSIFIED RESTARTS
# ==========================================

name = "matching-ii"
target = root / "results" / f"submission_{name}.json"

rows = load_transfers(root / "data" / f"{name}.txt")
best = read_incumbent(target, len(rows), name)

assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))
p = pack(rows)

# Ruajme rekordin para optimizimit
backup = target.parent / f"before_restarts_{name}_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

# Seed i ri per eksplorim tjeter
rng = random.Random(2509)


def shake(selected, n_drop):

    removed = set(
        rng.sample(
            sorted(selected),
            min(n_drop, len(selected))
        )
    )

    kept = set(selected) - removed

    oe, ol, od = owner_maps(p, kept)

    pool = set(removed)

    for j in removed:
        pool.update(p.e_rows[p.e[j]])
        pool.update(p.l_rows[p.l[j]])
        pool.update(p.d_rows[p.d[j]])

    alternatives = [
        int(j)
        for j in pool
        if j not in selected and p.w[j] > 0
    ]

    priorities = [
        (
            float(p.w[j]) * rng.uniform(0.65, 1.35),
            j
        )
        for j in alternatives
    ]

    priorities.sort(reverse=True)

    # Shtojme transferime alternative
    for _, j in priorities:

        if (
            oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0
        ):

            kept.add(j)

            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    # Rikthejme transferimet origjinale kur ka vend
    for j in sorted(removed, key=lambda k: -p.w[k]):

        if (
            oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0
        ):

            kept.add(j)

            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    score_solution(rows, sorted(kept))

    return kept


# ==========================================
# 3 RESTARTE ME PARAMETRA TE RINJ
# ==========================================

for trial, n_drop in enumerate([350, 600, 1000], 1):

    print(f"\n===== RESTART {trial}/3 =====", flush=True)

    starting = shake(best, n_drop)

    starting_score = score_solution(
        rows,
        sorted(starting)
    )

    print(
        f"Alternative start: {starting_score:.3f}",
        flush=True
    )

    candidate, info = adaptive_lns(
        rows,
        starting,
        seconds=100,
        rounds=2500,
        seed=2509 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    score = score_solution(
        rows,
        sorted(candidate)
    )

    print(
        f"Rezultati i restartit: {score:.3f}",
        flush=True
    )

    print(
        f"Rekordi: {best_score:.3f}",
        flush=True
    )

    # Ruajme vetem rezultate me te mira
    if score > best_score + 1e-7:

        disk = read_incumbent(
            target,
            len(rows),
            name
        )

        disk_score = score_solution(
            rows,
            sorted(disk)
        )

        if score > disk_score + 1e-7:

            document = [{
                "decisionVector": make_vector(
                    len(rows),
                    sorted(candidate)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(
                    document,
                    separators=(",", ":")
                )
            )

            best = candidate
            best_score = score

            print(
                "REKORD I RI U RUAJT!",
                flush=True
            )

    else:
        print(
            "Pa rekord te ri.",
            flush=True
        )


# ==========================================
# REZULTATI PERFUNDIMTAR
# ==========================================

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import adaptive_lns
from src.cli import read_incumbent, save_atomic, CHALLENGE

assert "shake" in globals(), "Mungon funksioni shake nga qeliza e kaluar!"

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
best = read_incumbent(target, len(rows), name)

assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))

backup = target.parent / f"before_restart3_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

# Seed i ri per eksplorim
rng = random.Random(2709)

# Restarte me ndryshime me te medha
for trial, n_drop in enumerate([600, 1000, 1600], 1):

    print(f"\n===== RESTART {trial}/3 =====", flush=True)

    starting = shake(best, n_drop)
    starting_score = score_solution(rows, sorted(starting))

    print(f"Alternative start: {starting_score:.3f}", flush=True)

    candidate, info = adaptive_lns(
        rows,
        starting,
        seconds=120,
        rounds=3000,
        seed=2709 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    score = score_solution(rows, sorted(candidate))

    print(f"Rezultati i restartit: {score:.3f}", flush=True)
    print(f"Rekordi: {best_score:.3f}", flush=True)

    if score > best_score + 1e-7:

        disk = read_incumbent(target, len(rows), name)
        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:

            document = [{
                "decisionVector": make_vector(
                    len(rows), sorted(candidate)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = candidate
            best_score = score

            print("REKORD I RI U RUAJT!", flush=True)

    else:
        print("Pa rekord te ri.", flush=True)

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, adaptive_lns
from src.cli import read_incumbent, save_atomic, CHALLENGE

assert "shake" in globals(), "Ekzekuto fillimisht qelizen ku eshte funksioni shake!"

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))

# Backup para eksperimentit
backup = target.parent / f"before_restart_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

# Parametra te rinj
rng = random.Random(2911)

for trial, n_drop in enumerate([300, 600, 900], 1):

    print(f"\n===== RESTART {trial}/3 =====", flush=True)

    starting = shake(best, n_drop)

    print(
        f"Alternative start: {score_solution(rows, sorted(starting)):.3f}",
        flush=True
    )

    candidate, info = adaptive_lns(
        rows,
        starting,
        seconds=120,
        rounds=3000,
        seed=2911 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    score = score_solution(rows, sorted(candidate))

    print(f"Rezultati i restartit: {score:.3f}", flush=True)
    print(f"Rekordi: {best_score:.3f}", flush=True)

    if score > best_score + 1e-7:

        disk = read_incumbent(target, len(rows), name)
        assert disk is not None

        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:

            document = [{
                "decisionVector": make_vector(
                    len(rows), sorted(candidate)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = candidate
            best_score = score

            print("REKORD I RI U RUAJT!", flush=True)
        else:
            print("Rezultati ekzistues eshte me i mire.", flush=True)

    else:
        print("Pa rekord te ri.", flush=True)

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, adaptive_lns
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))

backup = target.parent / f"before_guided_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)


# ==========================================
# STRATEGJIA E RE: CONFLICT-GUIDED SHAKE
# ==========================================

def guided_shake(selected, n_drop, seed):

    rng = random.Random(seed)
    oe0, ol0, od0 = owner_maps(p, selected)

    sample = rng.sample(
        range(len(rows)),
        min(12000, len(rows))
    )

    challengers = []

    for j in sample:

        if j in selected or p.w[j] <= 0:
            continue

        conflicts = {
            int(x)
            for x in (
                oe0[p.e[j]],
                ol0[p.l[j]],
                od0[p.d[j]]
            )
            if x >= 0
        }

        if not conflicts:
            continue

        lost_weight = sum(
            float(p.w[k]) for k in conflicts
        )

        potential = (
            float(p.w[j])
            - lost_weight
            + rng.uniform(-5, 5)
        )

        challengers.append((potential, conflicts))

    challengers.sort(
        key=lambda item: item[0],
        reverse=True
    )

    removed = set()
    guided_target = int(n_drop * 0.75)

    # Largojme transferimet qe pengojne
    # kandidatet me premtues
    for _, conflicts in challengers:

        new = conflicts - removed

        remaining = guided_target - len(removed)

        if remaining <= 0:
            break

        if len(new) > remaining:
            new = set(
                rng.sample(sorted(new), remaining)
            )

        removed.update(new)

    # Shtojme pak rastesi per diversifikim
    available = sorted(selected - removed)
    extra = min(n_drop - len(removed), len(available))

    removed.update(
        rng.sample(available, extra)
    )

    kept = set(selected) - removed
    oe, ol, od = owner_maps(p, kept)

    pool = set(removed)

    for j in removed:
        pool.update(p.e_rows[p.e[j]])
        pool.update(p.l_rows[p.l[j]])
        pool.update(p.d_rows[p.d[j]])

    alternatives = [
        int(j)
        for j in pool
        if j not in selected and p.w[j] > 0
    ]

    priorities = [
        (
            float(p.w[j]) * rng.uniform(0.55, 1.55),
            j
        )
        for j in alternatives
    ]

    priorities.sort(reverse=True)

    for _, j in priorities:

        if (
            oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0
        ):
            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    for j in sorted(
        removed,
        key=lambda k: -p.w[k]
    ):

        if (
            oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0
        ):
            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    score_solution(rows, sorted(kept))
    return kept


# ==========================================
# 2 RESTARTE ME STRATEGJINE E RE
# ==========================================

for trial, n_drop in enumerate([400, 800], 1):

    print(f"\n===== GUIDED RESTART {trial}/2 =====", flush=True)

    starting = guided_shake(
        best,
        n_drop,
        seed=3109 + trial * 101
    )

    start_score = score_solution(rows, sorted(starting))

    print(f"Alternative start: {start_score:.3f}", flush=True)

    candidate, info = adaptive_lns(
        rows,
        starting,
        seconds=120,
        rounds=3000,
        seed=3209 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    score = score_solution(rows, sorted(candidate))

    print(f"Rezultati i restartit: {score:.3f}", flush=True)
    print(f"Rekordi aktual: {best_score:.3f}", flush=True)

    if score > best_score + 1e-7:

        disk = read_incumbent(
            target, len(rows), name
        )

        assert disk is not None
        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:

            document = [{
                "decisionVector": make_vector(
                    len(rows), sorted(candidate)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = candidate
            best_score = score

            print("REKORD I RI U RUAJT!", flush=True)
        else:
            print("Rezultati ne disk eshte me i mire.", flush=True)

    else:
        print("Pa rekord te ri.", flush=True)


print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
from collections import defaultdict
import os, sys, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))

backup = target.parent / f"before_smart_swap_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)


def find_best_swap(selected, top_k=20):
    oe, ol, od = owner_maps(p, selected)

    buckets = defaultdict(list)
    best_gain = 0.0000001
    best_add = set()
    best_remove = set()

    for j in range(len(rows)):
        if j in selected or p.w[j] <= 0:
            continue

        blockers = {
            int(x)
            for x in (
                oe[p.e[j]],
                ol[p.l[j]],
                od[p.d[j]]
            )
            if x >= 0
        }

        loss = sum(float(p.w[k]) for k in blockers)
        gain = float(p.w[j]) - loss

        # Nje transfer i ri
        if gain > best_gain:
            best_gain = gain
            best_add = {j}
            best_remove = blockers

        # Kandidatet per zevendesim te dyfishte
        record = (gain, j, blockers)

        for blocker in blockers:
            buckets[blocker].append(record)

    # Dy transfere te reja qe ndajne
    # te pakten nje konflikt ekzistues
    for candidates in buckets.values():
        candidates.sort(key=lambda x: x[0], reverse=True)
        candidates = candidates[:top_k]

        for a in range(len(candidates)):
            _, j, blockers_j = candidates[a]

            for b in range(a + 1, len(candidates)):
                _, k, blockers_k = candidates[b]

                if (
                    p.e[j] == p.e[k]
                    or p.l[j] == p.l[k]
                    or p.d[j] == p.d[k]
                ):
                    continue

                removed = blockers_j | blockers_k

                gain = (
                    float(p.w[j])
                    + float(p.w[k])
                    - sum(float(p.w[x]) for x in removed)
                )

                if gain > best_gain:
                    best_gain = gain
                    best_add = {j, k}
                    best_remove = removed

    return best_gain, best_add, best_remove


for iteration in range(1, 6):

    gain, added, removed = find_best_swap(best)

    if not added:
        print("Nuk u gjet zevendesim me fitim.")
        break

    candidate = (best - removed) | added
    score = score_solution(rows, sorted(candidate))

    if score <= best_score + 1e-7:
        print("Nuk u konfirmua permiresimi.")
        break

    best = candidate
    best_score = score

    document = [{
        "decisionVector": make_vector(
            len(rows), sorted(best)
        ),
        "problem": name,
        "challenge": CHALLENGE
    }]

    save_atomic(
        target,
        json.dumps(document, separators=(",", ":"))
    )

    print(
        f"SMART SWAP #{iteration}: "
        f"{best_score:.3f} | "
        f"gain: {gain:.3f}",
        flush=True
    )

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.focused_cli \
  --input data/matching-ii.txt \
  --problem matching-ii \
  --method milp \
  --seconds 150 \
  --rounds 40 \
  --sizes 300,550,900 \
  --caps 2000,3500,6000 \
  --per-round 4 \
  --seed 3361

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/Luna-Tomato-Logistics")

if (root / "src/focused_cli.py").exists():
    print("FOCUSED OPTIMIZER ESHTE GATI!")
else:
    print("MUNGON FOCUSED OPTIMIZER!")

In [ ]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.focused_cli \
  --input data/matching-ii.txt \
  --problem matching-ii \
  --method milp \
  --seconds 150 \
  --rounds 40 \
  --sizes 300,550,900 \
  --caps 2000,3500,6000 \
  --per-round 4 \
  --seed 3361

In [ ]:
from pathlib import Path
import os, sys, random, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, adaptive_lns
from src.focused_opt import focused_search
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon zgjidhja!"

best_score = score_solution(rows, sorted(best))

# Backup para eksperimentit
backup = target.parent / f"before_hybrid_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)


# =====================================
# CONFLICT-GUIDED SHAKE
# =====================================

def guided_shake_v2(selected, n_drop, seed):
    rng = random.Random(seed)

    oe0, ol0, od0 = owner_maps(p, selected)
    sample = rng.sample(
        range(len(rows)),
        min(12000, len(rows))
    )

    challengers = []

    for j in sample:
        if j in selected or p.w[j] <= 0:
            continue

        blockers = {
            int(x)
            for x in (
                oe0[p.e[j]],
                ol0[p.l[j]],
                od0[p.d[j]]
            )
            if x >= 0
        }

        if not blockers:
            continue

        loss = sum(float(p.w[k]) for k in blockers)

        priority = (
            float(p.w[j])
            - loss
            + rng.uniform(-3, 3)
        )

        challengers.append((priority, blockers))

    challengers.sort(key=lambda x: x[0], reverse=True)

    removed = set()
    target_guided = int(n_drop * 0.75)

    for _, blockers in challengers:
        remaining = target_guided - len(removed)

        if remaining <= 0:
            break

        new = sorted(blockers - removed)

        if len(new) > remaining:
            new = rng.sample(new, remaining)

        removed.update(new)

    available = sorted(selected - removed)
    extra = min(n_drop - len(removed), len(available))

    if extra > 0:
        removed.update(rng.sample(available, extra))

    kept = set(selected) - removed
    oe, ol, od = owner_maps(p, kept)

    pool = set(removed)

    for j in removed:
        pool.update(p.e_rows[p.e[j]])
        pool.update(p.l_rows[p.l[j]])
        pool.update(p.d_rows[p.d[j]])

    alternatives = [
        int(j)
        for j in pool
        if j not in selected and p.w[j] > 0
    ]

    alternatives.sort(
        key=lambda j: float(p.w[j]) * rng.uniform(0.6, 1.4),
        reverse=True
    )

    for j in alternatives:
        if (
            oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0
        ):
            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    for j in sorted(removed, key=lambda x: -p.w[x]):
        if (
            oe[p.e[j]] < 0
            and ol[p.l[j]] < 0
            and od[p.d[j]] < 0
        ):
            kept.add(j)
            oe[p.e[j]] = j
            ol[p.l[j]] = j
            od[p.d[j]] = j

    score_solution(rows, sorted(kept))
    return kept


# =====================================
# DY EKSPERIMENTE HYBRID
# =====================================

for trial, n_drop in enumerate([350, 700], 1):

    print(f"\n===== HYBRID RESTART {trial}/2 =====", flush=True)

    starting = guided_shake_v2(
        best,
        n_drop,
        seed=3509 + trial * 101
    )

    print(
        f"Alternative start: "
        f"{score_solution(rows, sorted(starting)):.3f}",
        flush=True
    )

    # Faza 1: STRONG LNS
    improved, lns_info = adaptive_lns(
        rows,
        starting,
        seconds=110,
        rounds=2800,
        seed=3609 + trial * 101,
        neighborhood_sizes=(550, 950, 1600),
        candidate_caps=(6000, 10000, 16000),
        per_round_seconds=1.5,
        verbose=False
    )

    lns_score = score_solution(rows, sorted(improved))

    print(f"Pas STRONG LNS: {lns_score:.3f}", flush=True)

    # Faza 2: MILP
    polished, milp_info = focused_search(
        rows,
        improved,
        method="milp",
        seconds=30,
        rounds=150,
        seed=3709 + trial * 101,
        sizes=(300, 550, 900),
        caps=(2000, 3500, 6000),
        per_round=2.0,
        verbose=False
    )

    score = score_solution(rows, sorted(polished))

    print(f"Pas MILP: {score:.3f}", flush=True)
    print(f"Rekordi: {best_score:.3f}", flush=True)

    if score > best_score + 1e-7:

        disk = read_incumbent(target, len(rows), name)
        assert disk is not None

        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:

            document = [{
                "decisionVector": make_vector(
                    len(rows),
                    sorted(polished)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = set(polished)
            best_score = score

            print("REKORD I RI U RUAJT!", flush=True)
        else:
            print("Rezultati ne disk eshte me i mire.", flush=True)

    else:
        print("Pa rekord te ri.", flush=True)


print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, shutil, time, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, adaptive_lns
from src.focused_opt import focused_search
from src.cli import read_incumbent, save_atomic, CHALLENGE

assert "guided_shake_v2" in globals(), (
    "Ekzekuto fillimisht qelizen e meparshme HYBRID!"
)

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon zgjidhja!"

best = set(best)
best_score = score_solution(rows, sorted(best))

backup = target.parent / f"before_hybrid_v2_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

# ========================================
# HYBRID V2 - 2 RESTARTE
# ========================================

for trial, n_drop in enumerate([600, 1100], 1):

    print(f"\n===== HYBRID V2 {trial}/2 =====", flush=True)

    starting = guided_shake_v2(
        best,
        n_drop,
        seed=4109 + trial * 113
    )

    start_score = score_solution(rows, sorted(starting))
    print(f"Alternative start: {start_score:.3f}", flush=True)

    # FAZA 1: STRONG LNS
    improved, lns_info = adaptive_lns(
        rows,
        starting,
        seconds=140,
        rounds=3500,
        seed=4209 + trial * 113,
        neighborhood_sizes=(550, 950, 1600, 2000),
        candidate_caps=(6000, 10000, 16000, 20000),
        per_round_seconds=1.5,
        verbose=False
    )

    lns_score = score_solution(rows, sorted(improved))

    print(f"Pas STRONG LNS: {lns_score:.3f}", flush=True)

    # FAZA 2: MILP
    polished, milp_info = focused_search(
        rows,
        improved,
        method="milp",
        seconds=45,
        rounds=180,
        seed=4309 + trial * 113,
        sizes=(300, 550, 900),
        caps=(2000, 3500, 6000),
        per_round=2.0,
        verbose=False
    )

    milp_score = score_solution(rows, sorted(polished))
    print(f"Pas MILP: {milp_score:.3f}", flush=True)

    # Zgjedhim rezultatin me te mire te ketij restarti
    if milp_score >= lns_score:
        candidate = set(polished)
        score = milp_score
    else:
        candidate = set(improved)
        score = lns_score

    disk = read_incumbent(target, len(rows), name)
    assert disk is not None

    disk_score = score_solution(rows, sorted(disk))

    if score > disk_score + 1e-7:

        document = [{
            "decisionVector": make_vector(
                len(rows),
                sorted(candidate)
            ),
            "problem": name,
            "challenge": CHALLENGE
        }]

        save_atomic(
            target,
            json.dumps(document, separators=(",", ":"))
        )

        best = candidate
        best_score = score

        print("REKORD I RI U RUAJT!", flush=True)

    else:
        best = set(disk)
        best_score = disk_score

        print("Pa rekord te ri.", flush=True)


print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"ESA FITNESS EXPECTED: {-best_score:.3f}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, random, time, shutil, json

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, _restricted_milp
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon zgjidhja!"
best = set(best)
best_score = score_solution(rows, sorted(best))

backup = target.parent / f"before_anchor_exchange_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

rng = random.Random(5801)
start = time.perf_counter()
improvements = 0
tries = 0
no_solution = 0

for step in range(1, 151):

    if time.perf_counter() - start >= 150:
        break

    oe, ol, od = owner_maps(p, best)

    # Gjejme transfere alternative premtuese
    sample = rng.sample(
        range(len(rows)),
        min(2500, len(rows))
    )

    anchors = []

    for j in sample:
        if j in best or p.w[j] <= 0:
            continue

        blocked = {
            int(k)
            for k in (oe[p.e[j]], ol[p.l[j]], od[p.d[j]])
            if k >= 0
        }

        if not blocked:
            continue

        loss = sum(float(p.w[k]) for k in blocked)
        potential = float(p.w[j]) - loss

        anchors.append((
            potential + rng.uniform(-2, 2),
            j,
            blocked
        ))

    if not anchors:
        break

    anchors.sort(reverse=True, key=lambda x: x[0])

    _, anchor, blockers = rng.choice(
        anchors[:min(35, len(anchors))]
    )

    # Zgjedhim transferet ekzistuese qe do
    # te lejohen te ndryshohen
    limit = rng.choice([20, 40, 80, 120])
    active = set(blockers)
    frontier = list(blockers)

    while len(active) < limit:

        if not frontier:
            remaining = list(best - active)
            if not remaining:
                break

            k = rng.choice(remaining)
            active.add(k)
            frontier.append(k)

        k = frontier.pop(rng.randrange(len(frontier)))

        neighbors = (
            p.e_rows[p.e[k]]
            + p.l_rows[p.l[k]]
            + p.d_rows[p.d[k]]
        )

        if len(neighbors) > 60:
            neighbors = rng.sample(neighbors, 60)

        for j in neighbors:

            for other in (
                oe[p.e[j]], ol[p.l[j]], od[p.d[j]]
            ):
                other = int(other)

                if other >= 0 and other not in active:
                    active.add(other)
                    frontier.append(other)

                    if len(active) >= limit:
                        break

            if len(active) >= limit:
                break

    # Ndertojme grupin e kandidateve
    pool = set(active)

    for j in active:
        pool.update(p.e_rows[p.e[j]])
        pool.update(p.l_rows[p.l[j]])
        pool.update(p.d_rows[p.d[j]])

    options = []

    for j in pool:

        if p.w[j] <= 0:
            continue

        blocked = {
            int(k)
            for k in (
                oe[p.e[j]], ol[p.l[j]], od[p.d[j]]
            )
            if k >= 0
        }

        if not blocked.issubset(active):
            continue

        loss = sum(float(p.w[k]) for k in blocked)

        priority = (
            float(p.w[j])
            - 0.52 * loss
            + rng.uniform(-1.5, 1.5)
        )

        if j not in active:
            options.append((priority, j))

    options.sort(reverse=True)

    cap = 1800
    allowance = max(0, cap - len(active))

    high = min(len(options), int(allowance * 0.75))

    chosen = [j for _, j in options[:high]]
    remaining = [j for _, j in options[high:]]

    chosen += rng.sample(
        remaining,
        min(len(remaining), allowance - len(chosen))
    )

    candidates = sorted(active | set(chosen))

    if len(candidates) <= len(active):
        continue

    time_left = 150 - (time.perf_counter() - start)

    if time_left < 0.3:
        break

    # Optimizimi MILP i nenproblemit
    tries += 1

    try:
        proposed = _restricted_milp(
            p,
            candidates,
            seconds=min(1.2, time_left),
            rel_gap=0.005
        )
    except Exception as exc:
        no_solution += 1
        if no_solution <= 2:
            print("MILP error:", repr(exc))
        continue

    if proposed is None:
        no_solution += 1
        continue

    trial = (best - active) | set(proposed)

    try:
        score = score_solution(rows, sorted(trial))
    except ValueError:
        continue

    # Pranojme vetem nje rekord te ri
    if score > best_score + 1e-7:

        disk = read_incumbent(
            target, len(rows), name
        )

        assert disk is not None

        disk_score = score_solution(rows, sorted(disk))

        if score > disk_score + 1e-7:

            document = [{
                "decisionVector": make_vector(
                    len(rows), sorted(trial)
                ),
                "problem": name,
                "challenge": CHALLENGE
            }]

            save_atomic(
                target,
                json.dumps(document, separators=(",", ":"))
            )

            best = trial
            best_score = score
            improvements += 1

            print(
                f"ANCHOR IMPROVEMENT #{improvements}: "
                f"{best_score:.3f} | round {step}",
                flush=True
            )

    if step % 30 == 0:
        print(
            f"Progress {step}/150 | best={best_score:.3f}",
            flush=True
        )

print("\n============================")
print(f"WINNER SCORE: {best_score:.3f}")
print(f"IMPROVEMENTS: {improvements}")
print(f"MILP CALLS: {tries}")
print(f"NO SOLUTION: {no_solution}")
print("============================")

In [ ]:
from pathlib import Path
import os, sys, shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")
os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

print("===== VERIFIKIMI I PROJEKTIT =====")

for name in ["matching-i", "matching-ii"]:

    rows = load_transfers(root / "data" / f"{name}.txt")

    source = root / "results" / f"submission_{name}.json"

    chosen = read_incumbent(
        source,
        len(rows),
        name
    )

    assert chosen is not None, f"Mungon {name}!"

    score = score_solution(rows, sorted(chosen))

    print(f"{name}: {score:.3f} | VALID LOKAL")

    # Kopje e JSON-it per dorezim
    shutil.copy2(
        source,
        Path("/kaggle/working") / source.name
    )

# Backup komplet i projektit
backup = shutil.make_archive(
    "/kaggle/working/ESA_BACKUP_LATEST",
    "zip",
    root_dir="/kaggle/working",
    base_dir="Luna-Tomato-Logistics"
)

print("\nBACKUP U KRIJUA!")
print("FAJLLI:", backup)
print("GATI PER DOREZIM NE ESA!")

In [1]:
from pathlib import Path
from zipfile import ZipFile
import shutil

root = Path("/kaggle/working/Luna-Tomato-Logistics")
inp = Path("/kaggle/input")

assert (root / "src/strong_opt.py").exists(), "Rikthe projektin!"

names = [
    "src/pool_merge_cli.py",
    "tests/test_pool_merge.py"
]

found = [
    p for p in inp.rglob("pool_merge_cli.py")
    if p.parent.name == "src"
]

if found:
    source = found[0].parent.parent

    for name in names:
        dest = root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / name, dest)

else:
    zips = list(inp.rglob("ESA_SpOC4_POOL_MERGE.zip"))
    assert zips, "Shto Dataset-in e ri te Add Input!"

    with ZipFile(zips[0]) as z:
        for name in names:
            dest = root / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(name))

print("POOL MERGE OPTIMIZER U INSTALUA!")

AssertionError: Rikthe projektin!

In [2]:
from pathlib import Path
from zipfile import ZipFile
import os
import sys

inp = Path("/kaggle/input")
working = Path("/kaggle/working")
root = working / "Luna-Tomato-Logistics"

# Kerkojme backup-in me te fundit
backups = list(inp.rglob("ESA_BACKUP_LATEST.zip"))
backups += list(working.glob("ESA_BACKUP_LATEST.zip"))

if not (root / "src/strong_opt.py").exists():

    assert backups, (
        "BACKUP-I I FUNDIT NUK U GJET! "
        "Shto output-in e Notebook-ut nga Add Input."
    )

    backup = backups[0]

    print("BACKUP U GJET:", backup)

    with ZipFile(backup) as z:
        assert z.testzip() is None, "ZIP-i ka problem!"
        z.extractall(working)

assert (root / "src/strong_opt.py").exists(), "Projekti nuk u rikthye!"

os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

print("\nPROJEKTI U RIKTHYE!\n")

scores = {}

for name in ["matching-i", "matching-ii"]:

    rows = load_transfers(root / "data" / f"{name}.txt")

    selected = read_incumbent(
        root / "results" / f"submission_{name}.json",
        len(rows),
        name
    )

    assert selected is not None

    score = score_solution(rows, sorted(selected))
    scores[name] = score

    print(f"{name}: {score:.3f} | VALID LOKAL")

assert scores["matching-ii"] >= 72150.512, (
    "KUJDES: Ky backup eshte me i vjeter se rekordi yne!"
)

print("\nGATI PER POOL MERGE OPTIMIZER!")

BACKUP U GJET: /kaggle/input/notebooks/oltamorina/esa-spoc4-luna-tomato-logistics/ESA_BACKUP_LATEST.zip

PROJEKTI U RIKTHYE!

matching-i: 33155.153 | VALID LOKAL
matching-ii: 72150.513 | VALID LOKAL

GATI PER POOL MERGE OPTIMIZER!


In [3]:
from pathlib import Path
from zipfile import ZipFile
import shutil
import os

root = Path("/kaggle/working/Luna-Tomato-Logistics")
inp = Path("/kaggle/input")

assert (root / "src/strong_opt.py").exists(), "Projekti mungon!"

names = [
    "src/pool_merge_cli.py",
    "tests/test_pool_merge.py"
]

# Kerkojme dataset-in e algoritmit te ri
found = [
    p for p in inp.rglob("pool_merge_cli.py")
    if p.parent.name == "src"
]

if found:
    source = found[0].parent.parent

    for name in names:
        dest = root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / name, dest)

else:
    zips = list(inp.rglob("ESA_SpOC4_POOL_MERGE.zip"))

    assert zips, (
        "MUNGON POOL MERGE DATASET! "
        "Shtoje nga Add Input -> Your Work."
    )

    with ZipFile(zips[0]) as z:
        for name in names:
            dest = root / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(name))

os.chdir(root)

print("POOL MERGE OPTIMIZER U INSTALUA!")

POOL MERGE OPTIMIZER U INSTALUA!


In [5]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m unittest discover -s tests -p "test_pool_merge.py" -v

/kaggle/working
test_merge_can_recombine_two_worse_parents (test_pool_merge.PoolMergeTest.test_merge_can_recombine_two_worse_parents) ... ok
test_merge_cap_keeps_incumbent (test_pool_merge.PoolMergeTest.test_merge_cap_keeps_incumbent) ... ok
test_merge_milp_not_worse (test_pool_merge.PoolMergeTest.test_merge_milp_not_worse) ... ok
test_perturb_feasible (test_pool_merge.PoolMergeTest.test_perturb_feasible) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.028s

OK


In [1]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.pool_merge_cli \
  --problem matching-ii \
  --target-score 73805.590 \
  --restarts 6 \
  --drop-sizes 450 900 1500 \
  --lns-seconds 90 \
  --merge-every 2 \
  --merge-seconds 100 \
  --polish-seconds 45 \
  --merge-method auto \
  --workers 2 \
  --seed 5909

[Errno 2] No such file or directory: '/kaggle/working/Luna-Tomato-Logistics'
/kaggle/working
/usr/bin/python3: Error while finding module specification for 'src.pool_merge_cli' (ModuleNotFoundError: No module named 'src')


In [2]:
from pathlib import Path
from zipfile import ZipFile
import os
import sys
import shutil

inp = Path("/kaggle/input")
working = Path("/kaggle/working")
root = working / "Luna-Tomato-Logistics"

# Rikthe projektin nga backup-i i fundit
if not (root / "src/strong_opt.py").exists():

    backups = list(inp.rglob("ESA_BACKUP_LATEST.zip"))

    assert backups, (
        "BACKUP-I NUK U GJET! "
        "Shto versionin e ruajtur te Notebook-ut te Add Input."
    )

    with ZipFile(backups[0]) as z:
        assert z.testzip() is None
        z.extractall(working)

    print("PROJEKTI U RIKTHYE!")

assert (root / "src/strong_opt.py").exists()

# Rikthe algoritmin Pool Merge
module = root / "src/pool_merge_cli.py"

if not module.exists():

    extracted = [
        p for p in inp.rglob("pool_merge_cli.py")
        if p.parent.name == "src"
    ]

    if extracted:
        source = extracted[0].parent.parent

        for name in [
            "src/pool_merge_cli.py",
            "tests/test_pool_merge.py"
        ]:
            src = source / name
            if src.exists():
                dest = root / name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dest)

    else:
        zips = list(inp.rglob("ESA_SpOC4_POOL_MERGE.zip"))

        assert zips, (
            "POOL MERGE NUK U GJET! "
            "Shto Dataset-in Pool Merge te Add Input."
        )

        with ZipFile(zips[0]) as z:
            for name in [
                "src/pool_merge_cli.py",
                "tests/test_pool_merge.py"
            ]:
                dest = root / name
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(z.read(name))

os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution
from src.cli import read_incumbent

for name in ["matching-i", "matching-ii"]:
    rows = load_transfers(root / "data" / f"{name}.txt")
    chosen = read_incumbent(
        root / "results" / f"submission_{name}.json",
        len(rows),
        name
    )
    assert chosen is not None
    score = score_solution(rows, sorted(chosen))
    print(f"{name}: {score:.3f} | VALID LOKAL")

assert (root / "src/pool_merge_cli.py").exists()

print("\nPOOL MERGE OPTIMIZER ESHTE GATI!")

PROJEKTI U RIKTHYE!
matching-i: 33155.153 | VALID LOKAL
matching-ii: 72150.513 | VALID LOKAL

POOL MERGE OPTIMIZER ESHTE GATI!


In [3]:
%cd /kaggle/working/Luna-Tomato-Logistics

!python -m src.pool_merge_cli \
  --problem matching-ii \
  --target-score 73805.590 \
  --restarts 6 \
  --drop-sizes 450 900 1500 \
  --lns-seconds 90 \
  --merge-every 2 \
  --merge-seconds 100 \
  --polish-seconds 45 \
  --merge-method auto \
  --workers 2 \
  --seed 5909

/kaggle/working
BASELINE: 72150.513 | backup: /kaggle/working/Luna-Tomato-Logistics/results/before_pool_merge_matching-ii_1794292268.json
PO ATTAKOJME SYNIMIN: 73805.59

RESTART 1/6: drop=450 | start=71883.963
POST LNS 1: 72183.338 | iterations=393
*** NEW RECORD [LNS restart 1]: 72183.338 | gain since start: 32.825

RESTART 2/6: drop=900 | start=71572.996
POST LNS 2: 72207.021 | iterations=416
*** NEW RECORD [LNS restart 2]: 72207.021 | gain since start: 56.508
MERGE 2: 72211.209 | {'method': 'milp', 'variables': 9399, 'pool_members': 3, 'union_before_cap': 9399, 'status': 'FEASIBLE', 'seconds': 0.09361397700001817}
*** NEW RECORD [pool merge 2]: 72211.209 | gain since start: 60.696
POLISH 2: 72222.254
*** NEW RECORD [polish 2]: 72222.254 | gain since start: 71.741

RESTART 3/6: drop=1500 | start=71110.647
POST LNS 3: 72189.261 | iterations=414

RESTART 4/6: drop=450 | start=72002.861
POST LNS 4: 72239.034 | iterations=418
*** NEW RECORD [LNS restart 4]: 72239.034 | gain since start: 

In [4]:
from pathlib import Path
import os, sys, json, random, shutil, time

root = Path("/kaggle/working/Luna-Tomato-Logistics")
assert (root / "src/pool_merge_cli.py").exists(), "Rikthe projektin dhe Pool Merge!"

os.chdir(root)
sys.path.insert(0, str(root))

from src.matching import load_transfers, score_solution, make_vector
from src.strong_opt import pack, owner_maps, adaptive_lns, _restricted_milp
from src.pool_merge_cli import perturb
from src.cli import read_incumbent, save_atomic, CHALLENGE

name = "matching-ii"
target = root / "results/submission_matching-ii.json"

rows = load_transfers(root / "data/matching-ii.txt")
p = pack(rows)

best = read_incumbent(target, len(rows), name)
assert best is not None, "Mungon rezultati!"

best = set(best)
best_score = score_solution(rows, sorted(best))
initial_score = best_score

backup = target.parent / f"before_expanded_merge_{int(time.time())}.json"
shutil.copy2(target, backup)

print(f"REKORDI AKTUAL: {best_score:.3f}")
print(f"BACKUP: {backup}", flush=True)

# Lexojme zgjidhjet e vjetra nga backup-et
archive = [set(best)]

history = sorted(
    target.parent.glob("*.json"),
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

for file in history[:100]:
    if file == target:
        continue

    try:
        solution = read_incumbent(file, len(rows), name)
        if solution is None:
            continue

        value = score_solution(rows, sorted(solution))

        if value < best_score - 2500:
            continue

        if all(
            len(solution.symmetric_difference(other)) >= 45
            for other in archive
        ):
            archive.append(set(solution))

    except (ValueError, OSError, KeyError, TypeError):
        continue

    if len(archive) >= 18:
        break

print(f"ZGJIDHJE NE ARKIV: {len(archive)}", flush=True)

rng = random.Random(6809)


def accept(candidate, label):
    global best, best_score

    value = score_solution(rows, sorted(candidate))

    disk = read_incumbent(target, len(rows), name)
    assert disk is not None

    disk_score = score_solution(rows, sorted(disk))

    if value <= disk_score + 1e-7:
        best = set(disk)
        best_score = disk_score
        return False

    document = [{
        "decisionVector": make_vector(
            len(rows), sorted(candidate)
        ),
        "problem": name,
        "challenge": CHALLENGE
    }]

    save_atomic(
        target,
        json.dumps(document, separators=(",", ":"))
    )

    best = set(candidate)
    best_score = value

    print(
        f"*** REKORD I RI [{label}]: {best_score:.3f}",
        flush=True
    )

    return True


def expanded_merge(seed, cap=28000):
    local_rng = random.Random(seed)

    # Transferet nga zgjidhjet me te mira historike
    candidates = set(best)

    for solution in archive:
        candidates.update(solution)

    # Ruajme gjithmone zgjidhjen aktuale
    if len(candidates) > cap:
        extras = sorted(
            candidates - best,
            key=lambda j: float(p.w[j]),
            reverse=True
        )
        candidates = set(best) | set(
            extras[:cap - len(best)]
        )

    oe, ol, od = owner_maps(p, best)

    # Kerkojme transfere te reja ne dataset-in e plote
    ranked = []

    for j in range(len(rows)):
        if j in candidates or p.w[j] <= 0:
            continue

        blockers = {
            int(x)
            for x in (
                oe[p.e[j]],
                ol[p.l[j]],
                od[p.d[j]]
            )
            if x >= 0
        }

        loss = sum(float(p.w[k]) for k in blockers)

        quality = (
            float(p.w[j])
            - 0.42 * loss
            + local_rng.uniform(-1.5, 1.5)
        )

        ranked.append((quality, j))

    ranked.sort(reverse=True)

    slots = max(0, cap - len(candidates))

    # Kandidatet me premtues
    top_count = min(len(ranked), int(slots * 0.7))
    selected = [j for _, j in ranked[:top_count]]

    # Kandidatet shtese per diversitet
    remaining = [
        j for _, j in ranked[
            top_count:min(len(ranked), max(slots * 4, 1000))
        ]
    ]

    selected += local_rng.sample(
        remaining,
        min(len(remaining), slots - len(selected))
    )

    candidates.update(selected)
    ids = sorted(candidates)

    print(
        f"EXPANDED MERGE: {len(ids)} kandidatë",
        flush=True
    )

    solution = _restricted_milp(
        p,
        ids,
        seconds=100,
        rel_gap=0.003
    )

    return solution


# ==========================================
# TRI EKSPERIMENTE TE REJA
# ==========================================

for trial, n_drop in enumerate([600, 1100, 1800], 1):

    print(f"\n===== EXPANDED ROUND {trial}/3 =====", flush=True)

    source = best if trial != 2 else rng.choice(archive[:6])

    starting = perturb(p, source, n_drop, rng)

    print(
        f"Alternative start: "
        f"{score_solution(rows, sorted(starting)):.3f}",
        flush=True
    )

    # FAZA 1: STRONG LNS
    improved, info = adaptive_lns(
        rows,
        starting,
        seconds=100,
        rounds=3000,
        seed=6809 + trial * 113,
        neighborhood_sizes=(550, 950, 1600, 2300),
        candidate_caps=(6500, 10500, 16000, 22000),
        per_round_seconds=2.0,
        verbose=False
    )

    score = score_solution(rows, sorted(improved))
    print(f"PAS LNS: {score:.3f}", flush=True)

    archive.append(set(improved))
    accept(set(improved), f"LNS {trial}")

    # FAZA 2: Bashkimi i zgjeruar me MILP
    merged = expanded_merge(7109 + trial * 131)

    if merged is not None:
        merged_score = score_solution(rows, sorted(merged))

        print(
            f"PAS EXPANDED MERGE: {merged_score:.3f}",
            flush=True
        )

        accept(set(merged), f"MERGE {trial}")
    else:
        print("MILP nuk ktheu zgjidhje.", flush=True)

    # FAZA 3: Permiresimi perfundimtar me LNS
    polished, info = adaptive_lns(
        rows,
        best,
        seconds=50,
        rounds=1800,
        seed=7409 + trial * 131,
        neighborhood_sizes=(450, 900, 1600),
        candidate_caps=(5500, 10000, 18000),
        per_round_seconds=2.0,
        verbose=False
    )

    polish_score = score_solution(rows, sorted(polished))

    print(f"PAS POLISH: {polish_score:.3f}", flush=True)

    archive.append(set(polished))
    accept(set(polished), f"POLISH {trial}")

    print(f"REKORDI: {best_score:.3f}", flush=True)

    if best_score > 73805.590:
        print("TARGET SURPASSED LOCALLY!")
        break

# Verifikimi perfundimtar
final = read_incumbent(target, len(rows), name)
assert final is not None

final_score = score_solution(rows, sorted(final))

print("\n================================")
print(f"WINNER SCORE: {final_score:.3f}")
print(f"TOTAL GAIN: {final_score - initial_score:.3f}")
print(f"GAP TO TARGET: {73805.590 - final_score:.3f}")
print("VALID RESOURCE CONSTRAINTS: TRUE")
print("================================")

REKORDI AKTUAL: 72272.646
BACKUP: /kaggle/working/Luna-Tomato-Logistics/results/before_expanded_merge_1794293471.json
ZGJIDHJE NE ARKIV: 10

===== EXPANDED ROUND 1/3 =====
Alternative start: 71943.756
PAS LNS: 72281.503
*** REKORD I RI [LNS 1]: 72281.503
EXPANDED MERGE: 28000 kandidatë
PAS EXPANDED MERGE: 32071.221
PAS POLISH: 72286.256
*** REKORD I RI [POLISH 1]: 72286.256
REKORDI: 72286.256

===== EXPANDED ROUND 2/3 =====
Alternative start: 71234.305
PAS LNS: 72054.882
EXPANDED MERGE: 28000 kandidatë
PAS EXPANDED MERGE: 32165.893
PAS POLISH: 72289.737
*** REKORD I RI [POLISH 2]: 72289.737
REKORDI: 72289.737

===== EXPANDED ROUND 3/3 =====
Alternative start: 70806.370
PAS LNS: 72266.036
EXPANDED MERGE: 28000 kandidatë
PAS EXPANDED MERGE: 32208.257
PAS POLISH: 72291.849
*** REKORD I RI [POLISH 3]: 72291.849
REKORDI: 72291.849

WINNER SCORE: 72291.849
TOTAL GAIN: 19.203
GAP TO TARGET: 1513.741
VALID RESOURCE CONSTRAINTS: TRUE
